# Raw SciVer to Processed Evidence-Claim Mapping

This notebook shows how the fetched Hugging Face SciVer snapshot in `data/raw/SciVer` relates to the converted evidence-claim database artifacts in `data/processed`. It is meant as a bridge between raw dataset rows and the processed pair-level records used for retrieval and lightweight classification.

## 1. Setup

Resolve paths relative to the repository root. The notebook expects that `scripts/fetch_sciver.py` and `scripts/build_sciver_qdrant.py` have already been run.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

# Support execution from either repo root or notebooks/.
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "sciver_vector_db").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

from sciver_vector_db.parsing import find_split_files, load_split_examples, build_pair_records

RAW_ROOT = REPO_ROOT / "data/raw/SciVer"
PROCESSED_DIR = REPO_ROOT / "data/processed"
PAIRS_MANIFEST = PROCESSED_DIR / "sciver_pairs_manifest.parquet"
DOWNLOAD_MANIFEST = RAW_ROOT / "sciver_download_manifest.json"

print(f"Raw root: {RAW_ROOT}")
print(f"Pairs manifest: {PAIRS_MANIFEST}")

Raw root: /Users/conglongxu/Projects/Erdos-2016-Summer/codebase/data/raw/SciVer
Pairs manifest: /Users/conglongxu/Projects/Erdos-2016-Summer/codebase/data/processed/sciver_pairs_manifest.parquet


## 2. Inspect the Download Manifest

The fetch helper writes a manifest so users can confirm which Hugging Face dataset revision was downloaded and where the raw files were placed.

In [2]:
if DOWNLOAD_MANIFEST.exists():
    download_manifest = json.loads(DOWNLOAD_MANIFEST.read_text(encoding="utf-8"))
    # Print compact provenance fields instead of the full file list.
    manifest_summary = {key: download_manifest.get(key) for key in ["dataset_id", "revision", "local_path", "file_count", "split_files"]}
else:
    # A manually copied raw snapshot can still be mapped, but it lacks fetch provenance.
    manifest_summary = {"warning": "Download manifest not found; run scripts/fetch_sciver.py for provenance metadata."}
manifest_summary

{'warning': 'Download manifest not found; run scripts/fetch_sciver.py for provenance metadata.'}

## 3. Locate and Load Raw Split Files

The parser finds `valset.json` and `testset.json` without assuming a single exact directory layout. This is useful for Hugging Face snapshots and manually copied datasets.

In [3]:
split_files = find_split_files(RAW_ROOT)
examples, parser_skips = load_split_examples(RAW_ROOT, split_files)

print("Split files:")
for split, path in split_files.items():
    print(f"  {split}: {path}")
print(f"Loaded parsed examples: {len(examples):,}")
print(f"Parser skips: {dict(parser_skips)}")

Split files:
  val: /Users/conglongxu/Projects/Erdos-2016-Summer/codebase/data/raw/SciVer/valset.json
  test: /Users/conglongxu/Projects/Erdos-2016-Summer/codebase/data/raw/SciVer/testset.json
Loaded parsed examples: 3,000
Parser skips: {}


## 4. Examine One Raw Example

Raw rows can contain direct visual fields or multi-item fields such as `item1_type` and `item1_path`. The parser normalizes these into visual evidence items.

In [4]:
# Pick the first example that has at least one detected visual item.
raw_example = next(example for example in examples if example.visual_items)
visual_item = raw_example.visual_items[0]

raw_summary = {
    "split": raw_example.split,
    "paperid": raw_example.paperid,
    "request_id": raw_example.request_id,
    "claim": raw_example.claim,
    "label": raw_example.label,
    "claim_type": raw_example.claim_type,
    "detected_modality": visual_item.modality,
    "raw_image_path": visual_item.image_path_raw,
    "resolved_image_path": str(visual_item.image_path),
    "caption": visual_item.caption,
}
pd.Series(raw_summary)

split                                                                val
paperid                                                     2409.12320v1
request_id                                                             4
claim                  Despite stable medians at 5, the 3.5% decline ...
label                                                           entailed
claim_type                                                    analytical
detected_modality                                                  chart
raw_image_path                 ./SciVer/images/2409.12320v1_figure_4.png
resolved_image_path    /Users/conglongxu/Projects/Erdos-2016-Summer/c...
caption                                                                 
dtype: object

## 5. Load Processed Pair Records

The processed pairs manifest is one row per supervised chart/table-claim pair. It stores the label as metadata and target, while the embedded text used during vectorization excludes label, rationale, explanation, answer, and perturbation fields.

In [5]:
if not PAIRS_MANIFEST.exists():
    raise FileNotFoundError("Run scripts/build_sciver_qdrant.py before this notebook.")

pairs_df = pd.read_parquet(PAIRS_MANIFEST)
print(f"Processed pair records: {len(pairs_df):,}")
display(pairs_df.groupby(["modality", "label"]).size().rename("count").reset_index())
display(pairs_df.groupby("claim_type").size().rename("count").reset_index().sort_values("count", ascending=False))

Processed pair records: 1,500


,modality,label,count
0,chart,entailed,409
1,chart,refuted,408
2,table,entailed,341
3,table,refuted,342


,claim_type,count
0,analytical,750
1,direct,750


## 6. Map Raw Example to Processed IDs

The converter creates deterministic human-readable IDs for claims, evidence items, and pairs. The code below rebuilds pair records from the parsed raw examples and joins them to the processed manifest.

In [6]:
# Rebuild pair records in memory using the same default modalities as the build script.
pair_records, pair_skips = build_pair_records(examples, modalities={"chart", "table"})
rebuilt_df = pd.DataFrame(
    [
        {
            "pair_id": record.pair_id,
            "claim_id": record.claim_id,
            "evidence_item_id": record.evidence_item_id,
            "paperid": record.paperid,
            "request_id": record.request_id,
            "claim": record.claim,
            "modality": record.modality,
            "image_path": str(record.image_path),
        }
        for record in pair_records
    ]
)

# Join confirms that the processed manifest records come from deterministic raw parsing.
joined = rebuilt_df.merge(
    pairs_df[["pair_id", "split", "label", "label_id", "claim_type"]],
    on="pair_id",
    how="inner",
)
print(f"Rebuilt usable pairs: {len(rebuilt_df):,}")
print(f"Pair build skips: {dict(pair_skips)}")
print(f"Joined to processed manifest: {len(joined):,}")
joined.head(10)

Rebuilt usable pairs: 1,500
Pair build skips: {'multiple_matching_visual_evidence': 1500}
Joined to processed manifest: 0


,pair_id,claim_id,evidence_item_id,paperid,request_id,claim,modality,image_path,split,label,label_id,claim_type


## 7. Downstream Classifier Readiness

The processed database remains pair-centric: each row has vectors and metadata needed for simple classifiers. `label_id` is the target, while `claim_type` and `modality` can be used as filter fields or one-hot metadata features.

In [7]:
feature_export = PROCESSED_DIR / "sciver_pair_features.parquet"
if feature_export.exists():
    features_df = pd.read_parquet(feature_export)
    vector_columns = [column for column in ["claim_vec", "evidence_text_vec", "pair_text_vec", "image_vec"] if column in features_df.columns]
    print(f"Feature rows: {len(features_df):,}")
    print(f"Vector columns: {vector_columns}")
    display(features_df[["pair_id", "label", "label_id", "claim_type", "modality", *vector_columns]].head(3))
else:
    print("Feature export not found yet. Run scripts/export_pair_features.py after building Qdrant.")

Feature rows: 1,500
Vector columns: ['claim_vec', 'evidence_text_vec', 'pair_text_vec', 'image_vec']


,pair_id,label,label_id,claim_type,modality,claim_vec,evidence_text_vec,pair_text_vec,image_vec
0,pair:val:2410.01727v1:631:e591e8dfb5724723:231...,entailed,1,direct,table,"[-0.027619289234280586, -0.07258324325084686, ...","[-0.11883843690156937, 0.04829877242445946, -0...","[-0.01777704618871212, -0.0680219903588295, -0...","[0.016295723617076874, 0.010259835980832577, 0..."
1,pair:test:2410.21526v1:641:4bc58bc2b03b9c0e:a2...,refuted,0,analytical,table,"[-0.0051315282471477985, -0.08595585078001022,...","[-0.11883843690156937, 0.04829877242445946, -0...","[-0.0023410306312143803, -0.08794491738080978,...","[0.010177338495850563, 0.009773672558367252, -..."
2,pair:val:2410.22046v2:396:5fba8c88a57dc187:73d...,entailed,1,analytical,chart,"[-0.030919738113880157, -0.01596152037382126, ...","[-0.11883843690156937, 0.04829877242445946, -0...","[-0.025306956842541695, -0.012006157077848911,...","[0.032056841999292374, 0.011756455525755882, 0..."
